# Context Window Overflow Demo

**Based on IBM Research:** [Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1)

This notebook demonstrates how AI agents fail when tool outputs exceed the context window and how to fix it using the **Memory Pointer Pattern**.

---

## 🎯 What You'll Learn

1. **The Problem**: Large tool outputs overflow context windows
2. **The Solution**: Memory Pointer Pattern (7x token reduction)
3. **Implementation**: Using Strands Agents framework
4. **Real-World Use Case**: Log analysis system

---

## 📦 Setup

First, let's install dependencies and import required modules.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install strands-agents openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from strands import Agent
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from tools import (
    fetch_application_logs,
    analyze_error_patterns,
    detect_latency_anomalies,
    generate_incident_report
)

# Load environment variables
load_dotenv()

# Verify API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ OPENAI_API_KEY not set. Get your API key from https://platform.openai.com/api-keys and add OPENAI_API_KEY=your-key to a .env file.")

print("✅ Setup complete!")

---

## 🔬 Scenario 1: Baseline (No Context Management)

**Problem:** Agent tries to process large log dataset directly in context window.

**Expected:** Works but uses many tokens or may degrade performance.

In [ ]:
agent_baseline = Agent(
    model=OpenAIModel(model_id="gpt-4o-mini"),
    tools=[fetch_application_logs, analyze_error_patterns]
)

query = "Fetch 6 hours of logs for 'payment-service' and analyze error patterns. Report services with more than 20 errors."

response = agent_baseline(query)

tokens = len(query + str(response)) // 4
print(f"📊 Estimated tokens: {tokens:,}")

---

## 🧬 Scenario 2: Memory Pointer Pattern (IBM Research)

**Solution:** Store large data outside context, interact with pointers instead of raw data.

**Expected:** Success with ~7x token reduction (paper result).

In [ ]:
agent_pointer = Agent(
    model=OpenAIModel(model_id="gpt-4o-mini"),
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[
        fetch_application_logs,
        analyze_error_patterns,
        detect_latency_anomalies,
        generate_incident_report
    ]
)

query = "Fetch 12 hours of logs for 'payment-service', analyze error patterns and detect latency anomalies. Generate an incident report."

response = agent_pointer(query)

tokens = len(query + str(response)) // 4
print(f"📊 Estimated tokens: {tokens:,}")
if agent_pointer.state:
    for pointer in agent_pointer.state:
        size = len(str(agent_pointer.state[pointer]))
        print(f"  → '{pointer}': {size:,} bytes in agent.state (not in context)")

---

## 🎛️ Scenario 3: Custom Sliding Window

**Optimization:** Use smaller window size (20 messages) for more aggressive pruning.

In [ ]:
agent_custom = Agent(
    model=OpenAIModel(model_id="gpt-4o-mini"),
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    tools=[fetch_application_logs, analyze_error_patterns]
)

query = "Fetch 6 hours of logs for 'api-gateway' and analyze error patterns."

response = agent_custom(query)

tokens = len(query + str(response)) // 4
print(f"📊 Estimated tokens: {tokens:,}")

---

## 🔍 How Memory Pointer Pattern Works

When a tool fetches large data, instead of returning the raw bytes to the LLM:

1. Tool generates large data (e.g., 600 log events)
2. Tool checks if data exceeds a size threshold (~20KB)
3. If large: stores it in `agent.state` via `ToolContext`, returns only the pointer key
4. LLM receives a small string — `"logs-payment-service"` — not 145KB of JSON
5. The next tool reads the full data from `agent.state` using that key
6. Tool processes the complete dataset, writes results back as a new pointer

```python
@tool(context=True)
def fetch_application_logs(app_name: str, tool_context: ToolContext, hours: int = 6) -> str:
    logs = generate_logs(hours)                         # large data
    pointer = f"logs-{app_name}"
    tool_context.agent.state.set(pointer, logs)         # store outside context
    return f"Stored at: {pointer}"                      # only pointer enters LLM context
```

**Inspect `agent.state` from Scenario 2 below:**

In [ ]:
if agent_pointer.state:
    print(f"agent.state ({len(agent_pointer.state)} entries):")
    for pointer in list(agent_pointer.state)[:3]:
        data = agent_pointer.state[pointer]
        print(f"  {pointer}: {len(str(data)):,} bytes ({type(data).__name__})")

---

## 🧪 Interactive Experiment

Try your own queries! Modify the parameters below:

In [ ]:
HOURS = 3       # change: 1-24
APP_NAME = "auth-service"   # change: any service name

agent_experiment = Agent(
    model=OpenAIModel(model_id="gpt-4o-mini"),
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[fetch_application_logs, analyze_error_patterns, detect_latency_anomalies]
)

query = f"Fetch {HOURS} hours of logs for '{APP_NAME}', analyze errors and detect latency anomalies."

response = agent_experiment(query)

tokens = len(query + str(response)) // 4
print(f"📊 Estimated tokens: {tokens:,}  |  Memory pointers: {len(agent_experiment.state)}")

---

## 💬 Multi-turn Dialog: Querying Data from Memory

The agent fetches logs **once** on Turn 1 and stores them in `agent.state`. Turns 2 and 3 reuse the same pointer — the 145KB+ of data is never re-fetched or re-loaded into context.

| Turn | Query | What happens |
|------|-------|-------------|
| 1 | Fetch logs + analyze errors | Data stored in `agent.state["logs-payment-service"]` |
| 2 | Latency anomalies in those same logs | Reads from `agent.state` — no re-fetch |
| 3 | Which service had the most errors? | Reads from `agent.state` — no re-fetch |

In [ ]:
agent_dialog = Agent(
    model=OpenAIModel(model_id="gpt-4o-mini"),
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=[fetch_application_logs, analyze_error_patterns, detect_latency_anomalies]
)

# Turn 1: fetch + analyze
print("👤 Turn 1: Fetch 6 hours of logs for payment-service and analyze errors\n")
agent_dialog("Fetch 6 hours of logs for payment-service and analyze errors")

# Turn 2: reuses data already in agent.state
print("\n👤 Turn 2: Check latency anomalies in those same logs\n")
agent_dialog("Now check for latency anomalies in those same logs")

# Turn 3: still using same stored data
print("\n👤 Turn 3: Which service had the most errors?\n")
agent_dialog("Which service had the most errors?")

print(f"\n📦 agent.state: {len(agent_dialog.state._data)} pointer(s) — reused across all 3 turns")
for k, v in agent_dialog.state._data.items():
    print(f"  {k}: {len(str(v)):,} bytes")

---

## 📊 Summary & Key Findings

### Results from IBM Research Paper:

| Metric | Traditional | Memory Pointer | Improvement |
|--------|-------------|----------------|-------------|
| Tokens | 20,822,181 | 1,234 | **7x reduction** |
| Information Loss | ❌ Truncated | ✅ Complete | **100% preserved** |
| Execution | ❌ Failed | ✅ Success | **Workflow completed** |

### Key Takeaways:

1. ✅ **Memory Pointer Pattern eliminates context overflow** for large, indivisible datasets
2. ✅ **7x token reduction** without information loss (paper result)
3. ✅ **Transparent to agent** - no architectural changes needed
4. ✅ **Works with any tool** - simple wrapper pattern

### When to Use:

- ✅ Large tool outputs (>50KB)
- ✅ Indivisible data (logs, matrices, datasets)
- ✅ Multi-step workflows (output → input)
- ✅ Cost optimization (reduce tokens)

---

## 📚 Next Steps

1. ✅ Complete this demo
2. ➡️ Try [Demo 02: MCP Timeout](../02-mcp-timeout-demo/) - Handle external API timeouts
3. ➡️ Try [Demo 03: Reasoning Loops](../03-reasoning-loops-demo/) - Prevent infinite loops

---

## 🔗 Resources

- [IBM Research Paper](https://arxiv.org/html/2511.22729v1) - Original research
- [Strands Agents Documentation](https://strandsagents.com) - Framework docs
- [GitHub Repository](https://github.com/your-org/DevEx-Agent-Hallucinations) - Full code

---

In [ ]:
print("="*65)
print("BEFORE vs AFTER: Memory Pointer Pattern Impact")
print("="*65)

# agent_dialog ran 3 turns — data was fetched once and reused twice.
# Let's measure what actually stayed outside the LLM context.
if agent_dialog.state._data:
    total_bytes = sum(len(str(v)) for v in agent_dialog.state._data.values())
    pointer_bytes = sum(len(k) + 40 for k in agent_dialog.state._data.keys())

    est_tokens_without = total_bytes // 4
    est_tokens_with    = pointer_bytes // 4
    ratio      = est_tokens_without / max(est_tokens_with, 1)
    reduction  = (1 - est_tokens_with / est_tokens_without) * 100

    print(f"\n  {'':38} {'❌ Without pointer':>18} {'✅ With pointer':>16}")
    print("  " + "-"*72)
    print(f"  {'Data passed to LLM per call':38} {total_bytes:>16,} B {pointer_bytes:>14,} B")
    print(f"  {'Estimated tokens in context':38} {est_tokens_without:>18,} {est_tokens_with:>16,}")
    print(f"  {'Token reduction':38} {'baseline':>18} {f'{reduction:.0f}% fewer':>16}")
    print(f"  {'Ratio':38} {'1x':>18} {f'{ratio:.0f}x fewer':>16}")
    print()
    print(f"  What was stored in agent.state (never entered LLM context):")
    for pointer, data in agent_dialog.state._data.items():
        print(f"    • '{pointer}': {len(str(data)):,} bytes")
    print()
    print(f"  💡 IBM Research (arxiv 2511.22729) reported 7x reduction.")
    print(f"     This run shows ~{ratio:.0f}x — ratio scales with data volume.")
    print()
    print(f"  ✅ All 3 follow-up questions reused the same stored data.")
    print(f"     Zero re-fetches. Zero re-overflow. Just the pointer.")
else:
    print("\n⚠️  Run the Multi-turn Dialog cell first to populate agent.state.")